# LLM Wiki Walkthrough（llm-wiki.md 範式實作）

這份 notebook 把 [`llm-wiki.md`](./llm-wiki.md) 描述的**模式**變成可執行的最小版本。

## 跟 walkthrough.ipynb 的對照

| | `walkthrough.ipynb`（RAG） | 這份 notebook（Wiki） |
|---|---|---|
| 知識儲存 | 向量庫 + 圖譜（ChromaDB / SQLite） | `wiki/*.md` 純 markdown |
| 每次 query 做什麼 | 從 chunks 重新拼湊 | 讀已綜合好的 wiki 頁 |
| 知識累積 | 每筆 chunk 各自獨立 | 編譯一次 → 持續維護更新 |
| 主要操作 | upload / query | ingest / query / lint |
| 主角 | embedding model | LLM agent + schema 規範 |

## 工作目錄
本 notebook 會操作 **`./wiki-walkthrough-demo/`** 子資料夾（一個獨立的 sandbox），
不會動到 `wiki-project/`（那是 Stage 2 教學起點，README 寫明「等你動手」）。

跑完想重置：刪掉 `wiki-walkthrough-demo/` 整個資料夾即可。


## Step 0 — 環境準備

跟 `walkthrough.ipynb` 一樣，從 `.env` 載入 API key、注入 backend venv 的 site-packages（這樣不需要再 `uv add`，沿用 Stage 1 已裝好的 `google-generativeai`）。


In [2]:
import os, sys, pathlib

# 注入 backend venv 的 site-packages
BACKEND_VENV = pathlib.Path("backend/.venv").resolve()
candidates = list(BACKEND_VENV.glob("lib/python*/site-packages"))
assert candidates, f"找不到 backend venv: {BACKEND_VENV}"
site_pkgs = str(candidates[0])
if site_pkgs not in sys.path:
    sys.path.insert(0, site_pkgs)

# 從 .env 讀 GEMINI_API_KEY
env_path = pathlib.Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        

api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert api_key, "請在 .env 設定 GEMINI_API_KEY 或 GOOGLE_API_KEY"

import google.generativeai as genai
genai.configure(api_key=api_key)

# 兩個 model：純文字 + JSON 模式
gen_model = genai.GenerativeModel("gemini-2.5-flash")
json_model = genai.GenerativeModel(
    "gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"},
)
print("Gemini 設定完成")


ImportError: cannot import name 'cygrpc' from 'grpc._cython' (/Users/kevinluo/google-agent-ecosystem/Antigravity-work/Normal-RAG2Graph-Project/backend/.venv/lib/python3.13/site-packages/grpc/_cython/__init__.py)

## Step 1 — Bootstrap：建立三層架構

`llm-wiki.md` 的三層：

1. **`raw/`** — 原始來源（唯讀，LLM 只讀不寫）
2. **`wiki/`** — LLM 維護的 markdown 知識庫
3. **`AGENTS.md`** — schema，告訴 LLM 怎麼維護這份 wiki

這個 cell 把 sandbox 整個歸零並建好骨架。


In [2]:
import shutil, json
from pathlib import Path

ROOT = Path("wiki-walkthrough-demo")
if ROOT.exists():
    shutil.rmtree(ROOT)

(ROOT / "raw").mkdir(parents=True)
(ROOT / "wiki" / "sources").mkdir(parents=True)
(ROOT / "wiki" / "entities").mkdir(parents=True)
(ROOT / "wiki" / "concepts").mkdir(parents=True)
(ROOT / "wiki" / "synthesis").mkdir(parents=True)

# ---- AGENTS.md：給 LLM 看的 schema ----
AGENTS_MD = """# Wiki Schema（給 LLM 維護者）

## 目錄結構
- `raw/` — 原始來源，**LLM 只讀不寫**
- `wiki/sources/<slug>.md` — 每份來源的摘要頁
- `wiki/entities/<slug>.md` — 實體頁（公司、人物、產品、地點）
- `wiki/concepts/<slug>.md` — 概念頁（理念、技術、運動）
- `wiki/synthesis/<slug>.md` — 綜合分析頁（跨多來源的比較、論點）
- `wiki/index.md` — 全 wiki 目錄
- `wiki/log.md` — 時序日誌（append-only）

## 慣例
- slug 一律 kebab-case：`dario-amodei`、`ai-safety`
- 頁面之間用 `[[category/slug]]` 互連，例如 `[[entities/anthropic]]`
- 每頁開頭 frontmatter：

```yaml
---
title: ...
updated: YYYY-MM-DD
sources: [sources/article-x]
tags: [...]
---
```

## Ingest 流程
讀 source → 寫 `sources/<slug>.md` → 新增或更新 `entities/` `concepts/` 頁
→ 更新 `index.md` → append `log.md`

## 衝突處理
新舊事實衝突時，在頁面加 `> ⚠️ Conflict: ...` 區塊，**不直接覆蓋舊事實**。
"""

(ROOT / "AGENTS.md").write_text(AGENTS_MD, encoding="utf-8")

# ---- index.md / log.md 起手骨架 ----
(ROOT / "wiki" / "index.md").write_text(
    "# Wiki Index\n\n（由 LLM 維護；每次 ingest 後更新）\n\n"
    "## entities\n\n## concepts\n\n## sources\n\n## synthesis\n",
    encoding="utf-8",
)
(ROOT / "wiki" / "log.md").write_text(
    "# Wiki Log\n\n（append-only；格式：`## [YYYY-MM-DD] action | title`）\n\n",
    encoding="utf-8",
)

print("Bootstrap 完成：")
for p in sorted(ROOT.rglob("*")):
    rel = p.relative_to(ROOT)
    indent = "  " * len(rel.parts[:-1])
    print(f"  {indent}{rel.name}{'/' if p.is_dir() else ''}")


Bootstrap 完成：
  AGENTS.md
  raw/
  wiki/
    concepts/
    entities/
    index.md
    log.md
    sources/
    synthesis/


## Step 2 — 準備兩份 raw sources

刻意挑兩份**有交集**的文章，這樣 Step 4 的 ingest 才看得到「**LLM 更新既有頁面**」的行為。

- `source-1.md`：Anthropic 公司簡介（Anthropic / Dario Amodei / Claude / AI Safety）
- `source-2.md`：2026 AI 產業概況（提到 Anthropic、OpenAI、DeepMind、合作關係）


In [3]:
SOURCE_1 = """# 認識 Anthropic

Anthropic 是一家位於美國舊金山的人工智慧公司，由 Dario Amodei 與 Daniela Amodei
於 2021 年共同創立。創辦團隊大多曾任職於 OpenAI。

Anthropic 開發了 Claude 系列大型語言模型，包含 Claude 3、Claude 3.5 Sonnet，
以及最新的 Claude 4 系列。Anthropic 的研究方向以 AI 安全（AI Safety）與
可解釋性（Interpretability）為核心，著名的研究包含 Constitutional AI 與
Sleeper Agents 系列論文。
"""

SOURCE_2 = """# 2026 年 AI 產業概況

2026 年的生成式 AI 市場由三家領跑：Anthropic、OpenAI、Google DeepMind。

OpenAI 由 Sam Altman 擔任執行長，其代表產品為 ChatGPT 與 GPT 系列模型，
與 Microsoft 透過 Azure OpenAI Service 深度合作。

Google DeepMind 由 Demis Hassabis 領導，於 2023 年由 Google Brain 與 DeepMind
合併而成。其 Gemini 模型廣泛整合進 Google 產品線。

Anthropic 在企業市場成長迅速，與 Amazon AWS 達成深度合作，Claude 模型已成為
AWS Bedrock 的旗艦產品之一。Anthropic 估值於 2026 年初突破 600 億美元。
"""

(ROOT / "raw" / "source-1.md").write_text(SOURCE_1, encoding="utf-8")
(ROOT / "raw" / "source-2.md").write_text(SOURCE_2, encoding="utf-8")
print("已寫入 raw/source-1.md 與 raw/source-2.md")


已寫入 raw/source-1.md 與 raw/source-2.md


## Step 3 — Ingest pass 1

Ingest 函式做的事：

1. 把 schema (`AGENTS.md`) + 現有 wiki 狀態 + 新來源全部喂給 LLM
2. LLM 用 JSON 模式輸出操作清單：要寫哪些頁面（path + 完整 content）、要更新哪些 index 條目、log 一行
3. 我們 **依清單執行 file ops**，重建 `index.md`、append `log.md`

關鍵：每次 ingest 都把**現有 wiki 狀態完整餵給 LLM**——這就是 wiki 範式的成本（每次 ingest 是 O(wiki size)），但也是它能維持一致性的根本理由。


In [4]:
from datetime import date

def read_wiki_state(root: Path) -> str:
    """把目前 wiki/ 全部頁面打包成單一字串，餵給 LLM 當 context。"""
    lines = []
    schema = (root / "AGENTS.md").read_text(encoding="utf-8")
    lines.append(f"=== AGENTS.md ===\n{schema}")
    for p in sorted((root / "wiki").rglob("*.md")):
        rel = p.relative_to(root / "wiki").as_posix()
        lines.append(f"=== wiki/{rel} ===\n{p.read_text(encoding='utf-8')}")
    return "\n\n".join(lines)


INGEST_PROMPT_TPL = """你是一個 LLM Wiki 維護者。請依照下方 schema 跟既有 wiki 狀態，
處理一份新來源並產生 wiki 操作清單。

【schema 與既有 wiki 狀態】
{wiki_state}

【新來源】
路徑：raw/{source_filename}
內容：
{source_content}

請輸出 JSON，schema 如下：
{{
  "source_summary": "對這份來源的兩三句話摘要",
  "operations": [
    {{
      "path": "<category>/<slug>.md",
      "content": "完整 markdown，含 frontmatter；若為更新既有頁面，請整合舊內容與新事實，並在衝突處加 > ⚠️ Conflict: ..."
    }}
  ],
  "index_updates": [
    {{
      "category": "entities | concepts | sources | synthesis",
      "title": "頁面標題",
      "path": "<category>/<slug>.md",
      "summary": "一行摘要"
    }}
  ],
  "log_entry": "ingest: 來源標題 — 影響 N 個頁面"
}}

【路徑規則】（**很重要**）
- `path` 一律是 `<category>/<slug>.md` 的形式，例如 `entities/anthropic.md`、`sources/2026-ai-industry.md`
- **不要**寫成 `wiki/entities/anthropic.md`（不要加 `wiki/` 前綴）
- **不要**寫成 `/entities/anthropic.md`（不要開頭斜線）
- category 必須是 `entities` / `concepts` / `sources` / `synthesis` 其中之一

【內容規則】
- 每份來源**至少**要產生一個 sources/<slug>.md 摘要頁
- 來源中提到的關鍵實體（公司、人物、產品）必須對應到 entities/<slug>.md
- 頁面互連時用 `[[<category>/<slug>]]` 形式（無 .md 結尾，可加 alias `[[entities/anthropic|Anthropic]]`）
- 既有頁面若被觸及，operations 必須包含完整新版內容（會整份覆蓋）
- 不要重複建立既有頁面，更新就好
- **不要把 `path` 設為 `index.md` 或 `log.md`**——這兩個檔案由系統管理，LLM 不應寫入
"""


def _normalize_op_path(p: str) -> str:
    """容錯：去掉 LLM 可能多加的 'wiki/' 前綴或開頭斜線。"""
    p = p.strip().lstrip("/")
    if p.startswith("wiki/"):
        p = p[len("wiki/"):]
    return p


def rebuild_index(root: Path, index_updates_history: list[dict]) -> None:
    """把累積的 index_updates 重新組成 index.md（後寫的覆蓋同 path 的舊條目）。"""
    by_path = {}
    for entry in index_updates_history:
        path = _normalize_op_path(entry.get("path", ""))
        if not path:
            continue
        by_path[path] = {**entry, "path": path}
    by_cat = {"entities": [], "concepts": [], "sources": [], "synthesis": []}
    for entry in by_path.values():
        cat = entry.get("category", "sources")
        if cat in by_cat:
            by_cat[cat].append(entry)
    out = ["# Wiki Index", "", "（由 LLM 維護；每次 ingest 後更新）", ""]
    for cat in ["entities", "concepts", "sources", "synthesis"]:
        out.append(f"## {cat}")
        out.append("")
        for e in sorted(by_cat[cat], key=lambda x: x["path"]):
            out.append(f"- [[{cat}/{Path(e['path']).stem}]] — {e.get('summary', '')}")
        out.append("")
    (root / "wiki" / "index.md").write_text("\n".join(out), encoding="utf-8")


def append_log(root: Path, action: str, entry: str) -> None:
    line = f"## [{date.today().isoformat()}] {action} | {entry}\n\n"
    log_path = root / "wiki" / "log.md"
    log_path.write_text(log_path.read_text(encoding="utf-8") + line, encoding="utf-8")


INDEX_HISTORY: list[dict] = []


def ingest(root: Path, source_filename: str) -> dict:
    source_content = (root / "raw" / source_filename).read_text(encoding="utf-8")
    wiki_state = read_wiki_state(root)
    prompt = INGEST_PROMPT_TPL.format(
        wiki_state=wiki_state,
        source_filename=source_filename,
        source_content=source_content,
    )
    res = json_model.generate_content(prompt)
    plan = json.loads(res.text)

    # 執行 file ops（path 做防禦性 normalize；保護 index.md / log.md 不被 LLM 覆蓋）
    PROTECTED = {"index.md", "log.md"}
    for op in plan["operations"]:
        norm = _normalize_op_path(op["path"])
        op["path"] = norm
        if norm in PROTECTED or "/" not in norm:
            print(f"  [skip] LLM 嘗試寫入受保護檔案：{norm}")
            continue
        path = root / "wiki" / norm
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(op["content"], encoding="utf-8")

    INDEX_HISTORY.extend(plan.get("index_updates", []))
    rebuild_index(root, INDEX_HISTORY)
    append_log(root, "ingest", plan.get("log_entry", source_filename))
    return plan


plan1 = ingest(ROOT, "source-1.md")
print("source_summary:", plan1["source_summary"])
print()
print("operations:")
for op in plan1["operations"]:
    print(f"  - {op['path']}  ({len(op['content'])} chars)")
print()
print("index_updates:")
for e in plan1["index_updates"]:
    print(f"  - [{e['category']}] {e['title']} → {e['path']}")
print()
print("log_entry:", plan1["log_entry"])


NameError: name 'json_model' is not defined

## Step 4 — Ingest pass 2（看 LLM 更新既有頁面）

這次 LLM 看得到 `entities/anthropic.md` 已經存在；好的維護者應該：

- 把第二份來源的新事實（AWS 合作、估值 600 億）**整合進**既有 Anthropic 頁
- 為新出現的實體（OpenAI、DeepMind、Sam Altman 等）建立新頁
- 若 source-2 跟 source-1 衝突，加 `> ⚠️ Conflict:` 區塊（這份 demo 兩份來源沒有衝突，但機制保留）

跑完看 `wiki/entities/anthropic.md` 應該變成「source-1 + source-2 的綜合版」，而不是整份被覆蓋。


In [ ]:
plan2 = ingest(ROOT, "source-2.md")
print("source_summary:", plan2["source_summary"])
print()
print("operations:")
for op in plan2["operations"]:
    path = ROOT / "wiki" / op["path"]
    is_update = path.exists() and (path.read_text(encoding="utf-8") == op["content"])
    print(f"  - {op['path']}  ({len(op['content'])} chars)")
print()
print("index_updates:")
for e in plan2["index_updates"]:
    print(f"  - [{e['category']}] {e['title']} → {e['path']}")
print()

# 顯示更新後的 Anthropic 頁
anthropic = ROOT / "wiki" / "entities" / "anthropic.md"
if anthropic.exists():
    print("=" * 60)
    print(f"更新後的 {anthropic.relative_to(ROOT)}：")
    print("=" * 60)
    print(anthropic.read_text(encoding="utf-8"))


## Step 5 — Query

Wiki 範式的 query：

1. **先讀 `index.md`** → LLM 列出可能相關的頁面 path
2. **fan-out 讀取**那些頁面
3. **綜合**出帶引用的回答

這比 RAG 多一層 LLM hop，但**模型看的是已經被結構化、彼此連結、人類可讀的綜合內容**，而不是 chunked 原始片段。回答品質與引用準確度通常更高。


In [ ]:
QUERY_SELECT_TPL = """以下是一份 wiki 的 index.md：

{index_md}

使用者問題：{question}

請從 index 中挑選最相關的頁面，用 JSON 回傳。
**規則**：
- 挑 2~5 個頁面；除了實體頁外，**也要包含可能寫到答案的 sources/ 頁面**
- `pages` 內每一項必須是完整路徑，例如 `"entities/anthropic.md"`、`"sources/2026-ai-industry.md"`
- 不要只寫 slug 或標題；不要省略 `.md`
- 只能挑 index 中真實列出的頁面，不可虛構

格式：
{{
  "pages": ["entities/anthropic.md", "sources/2026-ai-industry.md"],
  "reason": "為什麼選這些"
}}
"""

QUERY_ANSWER_TPL = """你是一個精準的問答助手。**只根據以下 wiki 頁面內容**回答問題；
若 wiki 頁面沒有寫到，請誠實回答「wiki 中沒有這個資訊」，**不要使用你的訓練知識補充**。
一律使用繁體中文回答。回答中每個事實後標註來源，例如 `[[entities/anthropic]]`。

【wiki 頁面】
{pages}

使用者問題：{question}

請回答："""


def _normalize_page_path(p: str) -> str:
    """容錯處理：LLM 可能回 'entities/anthropic'、'AWS'、'wiki/entities/anthropic.md'。"""
    p = p.strip().strip("[]").split("|", 1)[0].strip().lstrip("/")
    if p.startswith("wiki/"):
        p = p[len("wiki/"):]
    if not p.endswith(".md"):
        p = p + ".md"
    return p


def query(root: Path, question: str) -> dict:
    index_md = (root / "wiki" / "index.md").read_text(encoding="utf-8")

    res = json_model.generate_content(QUERY_SELECT_TPL.format(
        index_md=index_md, question=question,
    ))
    selection = json.loads(res.text)
    raw_pages = selection.get("pages", [])

    selected, skipped = [], []
    for raw in raw_pages:
        norm = _normalize_page_path(raw)
        if (root / "wiki" / norm).exists():
            selected.append(norm)
        else:
            skipped.append(raw)

    pages_text = []
    for pg in selected:
        pages_text.append(f"=== {pg} ===\n{(root / 'wiki' / pg).read_text(encoding='utf-8')}")

    if not pages_text:
        return {
            "selected_pages": [],
            "selection_reason": selection.get("reason", ""),
            "skipped": skipped,
            "answer": "（沒有挑到任何存在的頁面，無法回答。）",
        }

    res2 = gen_model.generate_content(QUERY_ANSWER_TPL.format(
        pages="\n\n".join(pages_text),
        question=question,
    ))
    return {
        "selected_pages": selected,
        "selection_reason": selection.get("reason", ""),
        "skipped": skipped,
        "answer": res2.text,
    }


QUESTION = "Anthropic 是誰創立的？跟 AWS 是什麼關係？"
result = query(ROOT, QUESTION)

print(f"問題：{QUESTION}\n")
print(f"LLM 挑選的頁面：{result['selected_pages']}")
if result.get("skipped"):
    print(f"略過（路徑無法解析或不存在）：{result['skipped']}")
print(f"挑選理由：{result['selection_reason']}\n")
print("=" * 60)
print("回答：")
print("=" * 60)
print(result["answer"])


## Step 6 — Lint（健檢）

`llm-wiki.md` 提到的 lint 檢查：孤兒頁、缺反向連結、被多次提及但無頁面的概念、矛盾標記等。

這個 cell 做兩個機械式檢查（不需要 LLM）：

1. **孤兒頁面**：沒有任何其他頁面用 `[[category/slug]]` 指向它
2. **斷掉的連結**：頁面 A 引用 `[[xxx]]` 但 `xxx.md` 不存在

機械式 lint 跑完，再可以丟給 LLM 做語意層面的健檢（這裡示範前者）。


In [ ]:
import re

WIKI_LINK_RE = re.compile(r"\[\[([^\]]+)\]\]")


def _parse_link(raw: str) -> str:
    """處理 [[entities/anthropic]] / [[entities/anthropic|Anthropic]] / [[entities/anthropic.md]]
    一律回傳『不含 .md、不含 alias』的標準 slug：'entities/anthropic'。"""
    target = raw.strip().split("|", 1)[0].strip()
    if target.endswith(".md"):
        target = target[:-3]
    return target


def lint(root: Path) -> dict:
    wiki_dir = root / "wiki"
    pages = {
        p.relative_to(wiki_dir).with_suffix("").as_posix(): p
        for p in wiki_dir.rglob("*.md")
        if p.name not in ("index.md", "log.md")
    }

    incoming: dict[str, set[str]] = {k: set() for k in pages}
    broken: list[tuple[str, str]] = []

    for slug, path in pages.items():
        content = path.read_text(encoding="utf-8")
        for m in WIKI_LINK_RE.finditer(content):
            target = _parse_link(m.group(1))
            if target in pages:
                if target != slug:
                    incoming[target].add(slug)
            else:
                broken.append((slug, target))

    orphans = sorted([k for k, v in incoming.items() if not v])
    return {
        "orphans": orphans,
        "broken_links": broken,
        "incoming_count": {k: len(v) for k, v in incoming.items()},
    }


report = lint(ROOT)
print("=== Lint Report ===\n")
print("入度（被多少頁面引用）：")
for slug, n in sorted(report["incoming_count"].items(), key=lambda x: -x[1]):
    print(f"  {n:>2}  {slug}")
print()
print(f"孤兒頁面（入度 = 0）：{len(report['orphans'])}")
for o in report["orphans"]:
    print(f"  - {o}")
print()
print(f"斷掉的連結：{len(report['broken_links'])}")
for src, tgt in report["broken_links"]:
    print(f"  {src}.md  →  [[{tgt}]]  (target 不存在)")


## Step 7 — 把 query 結果歸檔回 wiki

`llm-wiki.md` 的關鍵洞察：「**好的回答應該被歸檔回 wiki 變成新頁面**」，否則探索成果會消失在對話記錄裡。

這個 cell 把 Step 5 的回答寫成一頁 `synthesis/<slug>.md`，並更新 index 與 log。
下次 ingest 新來源時，LLM 也看得到這份 synthesis，可以引用或更新它。


In [ ]:
import re as _re

def slugify(text: str) -> str:
    text = _re.sub(r"[^\w\s-]", "", text.lower())
    text = _re.sub(r"[\s_]+", "-", text).strip("-")
    return text[:60] or "untitled"


def archive_query_to_wiki(root: Path, question: str, result: dict) -> Path:
    slug = slugify(question)
    rel_path = f"synthesis/{slug}.md"
    page = f"""---
title: {question}
updated: {date.today().isoformat()}
sources: {result["selected_pages"]}
tags: [query-archive]
---

# {question}

{result["answer"]}

## 引用的頁面
""" + "\n".join(f"- [[{Path(p).with_suffix('').as_posix()}]]" for p in result["selected_pages"]) + "\n"

    target = root / "wiki" / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(page, encoding="utf-8")

    INDEX_HISTORY.append({
        "category": "synthesis",
        "title": question,
        "path": rel_path,
        "summary": (result["answer"][:80].replace("\n", " ") + "..."),
    })
    rebuild_index(root, INDEX_HISTORY)
    append_log(root, "archive", f"query → {rel_path}")
    return target


archived = archive_query_to_wiki(ROOT, QUESTION, result)
print(f"已歸檔：{archived.relative_to(ROOT)}")
print()
print("=" * 60)
print(archived.read_text(encoding="utf-8"))


## Step 8 — 看一下最終 wiki 形狀

跑完所有步驟後，sandbox 應該包含：raw 兩份、wiki 一份 index、一份 log、若干 entities/sources/synthesis 頁面。


In [ ]:
print("=" * 60)
print(f"{ROOT}/ 結構：")
print("=" * 60)
for p in sorted(ROOT.rglob("*")):
    if p.is_dir():
        continue
    rel = p.relative_to(ROOT)
    size = p.stat().st_size
    print(f"  {str(rel):<50}  {size:>6} bytes")

print()
print("=" * 60)
print("wiki/index.md：")
print("=" * 60)
print((ROOT / "wiki" / "index.md").read_text(encoding="utf-8"))

print("=" * 60)
print("wiki/log.md：")
print("=" * 60)
print((ROOT / "wiki" / "log.md").read_text(encoding="utf-8"))


## 收工 — 三份檔案的對位

| 檔案 | 路線 | 給誰看 |
|---|---|---|
| `walkthrough.ipynb` | RAG + GraphRAG（向量庫 + 圖譜） | 想理解 Stage 1 後端內部資料流的學員 |
| `wiki-walkthrough.ipynb`（這份） | LLM Wiki（markdown + LLM 維護） | 想體會 `llm-wiki.md` 範式怎麼真的跑起來的學員 |
| `wiki-project/`（不在 notebook 裡） | Stage 2 教學起點（空、等學員動手） | 跟 Antigravity 共創自己的 schema 的學員 |

## 為什麼這份 notebook 用 sandbox 而不是直接動 `wiki-project/`？
保留 `wiki-project/` 的「等你動手」精神。看完這份 notebook 後，學員應該：

1. **理解模式**怎麼運作
2. **進入 `wiki-project/`** 跟 Antigravity 共創自己的 `CLAUDE.md` schema
3. 把這份 notebook 的 ingest / query / lint 函式**改寫成符合自己領域的版本**

## 下一步
- 想看更嚴謹的版本？把 `wiki-project/CLAUDE.md` 補完，把這份 notebook 的函式搬進 `wiki-project/scripts/`
- 想換成 LangChain？切到 `feature/semantic-graphrag` 分支，看 `langchain_md/` 的對照組
- 跑完想清空：`rm -rf wiki-walkthrough-demo/`
